# Demo: an agent that scrapes for itself

*Instructor demo: watch, don't type.*

In `llm_analysis.ipynb` **we** decided everything: which pages to download, which text to send, which question to ask. The model only answered.

An **agent** is the same model in a loop, with one extra power: it can ask *us* to run a tool for it (here: download a web page). We run the tool, give it the result, and ask again, until it says it is done.

```
question ──► model ──► "please fetch URL X" ──► Python downloads X ──┐
               ▲                                                     │
               └────────────── page text ◄───────────────────────────┘
             ... until the model gives a final answer
```

In [ ]:
import requests, json
from getpass import getpass
from bs4 import BeautifulSoup

api_key = getpass("GPUStack API key: ")
base_url = "https://gpustack.unibe.ch/v1"
model = "qwen3.8-27b"   # slower but more careful; gpt-oss-120b also supports tools

## The tool: Lecture 8, unchanged

`fetch_page(url)` is `extract_text()` from Lecture 8, plus the download. It returns the text, or the error as text, so the model also sees failures.

In [ ]:
def extract_text(html):
    soup = BeautifulSoup(html, "html.parser")
    content = soup.find("main") or soup.find("article") or soup.find("body") or soup
    for element in content.find_all(["script", "style", "noscript", "nav", "header", "footer"]):
        element.decompose()
    texts = [p.get_text(" ", strip=True) for p in content.find_all("p")]
    return "\n".join(t for t in texts if t)

def fetch_page(url):
    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        text = extract_text(response.content)
        return text[:4000] if text else "The page has no paragraph text."
    except requests.RequestException as error:
        return "Download failed: " + str(error)

## Describe the tool to the model

The model never runs Python. It only sees this description and can reply "call `fetch_page` with this URL".

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "fetch_page",
        "description": "Download a web page and return its paragraph text (max 4000 characters), or an error message.",
        "parameters": {
            "type": "object",
            "properties": {"url": {"type": "string", "description": "The full URL"}},
            "required": ["url"],
        },
    },
}]

## The agent loop

Each round: send the whole conversation; if the model asks for a tool, run it, add the result to the conversation, repeat. We stop after 6 rounds no matter what: an agent without a limit can keep going (and keep downloading).

In [ ]:
def run_agent(task, max_rounds=6):
    messages = [{"role": "user", "content": task}]
    for round_number in range(max_rounds):
        data = {"model": model, "messages": messages, "tools": tools, "temperature": 0}
        response = requests.post(base_url + "/chat/completions",
                                 headers={"Authorization": "Bearer " + api_key},
                                 json=data, timeout=180)
        response.raise_for_status()
        message = response.json()["choices"][0]["message"]
        messages.append(message)

        if not message.get("tool_calls"):
            print("FINAL ANSWER:\n" + (message.get("content") or ""))
            return messages

        for call in message["tool_calls"]:
            url = json.loads(call["function"]["arguments"])["url"]
            result = fetch_page(url)
            print(f"[round {round_number + 1}] fetch_page({url}) -> {result[:120]!r}")
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})

    print("Stopped: too many rounds.")
    return messages

## Run it

The task names pages from the Lecture 8 list, including some that refused us (403). **Predict before running:** what will the agent do with a page it cannot download?

In [ ]:
task = ('Compare what these two organisations say about solar energy. '
        'Use the fetch_page tool to read their pages. If a page cannot be read, say so clearly '
        'and do not guess its content.\n'
        '1. https://energy.ec.europa.eu/topics/renewable-energy/solar-energy_en\n'
        '2. https://www.iaea.org/')

conversation = run_agent(task)

## Discuss

- Which pages did it decide to fetch? Did it fetch anything we did not ask for?
- The IAEA page refuses our requests (403 in Lecture 8). Did the agent say so, or did it describe the IAEA from memory?
- Compare with the notebook: there, *we* saw every row, every failure, and saved a table. Here we only see what we printed. **What would you need to log to trust an agent's answer in your thesis?**